# OTT Streaming Content Trend Analysis

Explore catalog mix, genre popularity, release-year trends, content ratings, country representation, and runtime. **The bundled data is synthetic and illustrative, not official streaming-platform data.**

## Business goal
Convert raw catalog metadata into actionable hypotheses for a content acquisition team.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent
raw = pd.read_csv(ROOT / 'data' / 'ott_catalog.csv')
print('Raw shape:', raw.shape)
display(raw.head())

## 1. Data quality audit
Identify duplicate IDs, missing values, and basic data types before analyzing.

In [ ]:
quality = pd.DataFrame({
    'dtype': raw.dtypes.astype(str),
    'missing_count': raw.isna().sum(),
    'missing_pct': (raw.isna().mean()*100).round(2),
    'unique_values': raw.nunique()
})
display(quality)
print('Duplicate show IDs:', raw['show_id'].duplicated().sum())

## 2. Clean and enrich the data
Remove duplicate IDs, standardize missing categorical values, and split duration into numeric value and unit.

In [ ]:
df = raw.drop_duplicates(subset=['show_id']).copy()
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
for col in ['director','country','rating','genre','type']:
    df[col] = df[col].fillna('Unknown')
df['duration_value'] = pd.to_numeric(df['duration'].astype(str).str.extract(r'(\\d+)')[0], errors='coerce')
df['duration_unit'] = np.where(df['duration'].astype(str).str.contains('Season', case=False), 'Seasons', 'Minutes')
df['date_added_year'] = df['date_added'].dt.year
display(df.head())
print('Cleaned shape:', df.shape)

## 3. Catalog composition
A first pass at the balance between movies and TV shows.

In [ ]:
type_counts = df['type'].value_counts()
ax = type_counts.plot(kind='bar', title='Catalog Composition by Content Type', ylabel='Titles', xlabel='Content type', rot=0)
plt.tight_layout(); plt.show()
display((type_counts / len(df) * 100).round(1).rename('share_pct').to_frame())

## 4. Genre popularity

In [ ]:
genre_counts = df['genre'].value_counts().sort_values()
genre_counts.plot(kind='barh', title='Titles by Primary Genre', xlabel='Titles', ylabel='Genre', figsize=(8,5)); plt.tight_layout(); plt.show()
display(df['genre'].value_counts().rename_axis('genre').reset_index(name='title_count'))

## 5. Release-year trend
Release year indicates when a title was released, not when the platform acquired it.

In [ ]:
df.groupby('release_year').size().plot(kind='line', marker='o', markersize=2, title='Titles by Release Year', xlabel='Release year', ylabel='Titles'); plt.tight_layout(); plt.show()

## 6. Regional representation

In [ ]:
country_counts = df.loc[df['country']!='Unknown','country'].value_counts().head(10).sort_values()
country_counts.plot(kind='barh', title='Top Countries Represented', xlabel='Titles', ylabel='Country', figsize=(8,5)); plt.tight_layout(); plt.show()

## 7. Content ratings

In [ ]:
df['rating'].value_counts().sort_values().plot(kind='barh', title='Content Rating Distribution', xlabel='Titles', ylabel='Rating', figsize=(8,5)); plt.tight_layout(); plt.show()

## 8. Movie runtime

In [ ]:
movie_minutes = df.loc[(df['type']=='Movie') & (df['duration_unit']=='Minutes'), 'duration_value'].dropna()
movie_minutes.plot(kind='hist', bins=20, title='Movie Runtime Distribution', xlabel='Runtime (minutes)', ylabel='Number of movies'); plt.tight_layout(); plt.show()
print('Median runtime:', movie_minutes.median())

## 9. Genre mix by content type

In [ ]:
pd.crosstab(df['genre'], df['type']).plot(kind='bar', stacked=True, figsize=(9,5), title='Genre Mix by Content Type', ylabel='Titles', xlabel='Genre', rot=45); plt.tight_layout(); plt.show()

## 10. Summary and next steps
Use the companion `reports/content_strategy_memo.md` for three actionable hypotheses. Before making investment decisions, join catalog metadata to actual title-level engagement, retention, and cost data.